In [1]:
from astropy.table import Table
import pandas as pd
import numpy as np
from pandarallel import pandarallel
from concurrent.futures import ThreadPoolExecutor

In [2]:
# Initialize pandarallel to use all available CPUs (8 threads)
pandarallel.initialize(progress_bar=True, nb_workers=8)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [3]:

def load_votable_to_dataframe(file_path):

    # Read the VOTable file using astropy
    votable = Table.read(file_path, format='votable')

    # Convert to Pandas DataFrame
    df = votable.to_pandas()

    return df

# Example usage
file_path = "C:\\git_repo\\cool-dwarf_stellar_parameter_inference_from_survey_data\\data\\LAMOST_data\\LAMOST_G0-9_Gaia_joined.vot"
df = load_votable_to_dataframe(file_path)


In [4]:
df.head()

,lamost_g_dwarf_oid,obsid,gaia_source_id,gaia_dr3_id,Plx,e_Plx,Gmag,BPmag,RPmag
0,1889706,482907076,12751758735104,12751758735104,0.837770,0.078726,16.563858,17.084002,15.898994
1,1891666,483007065,21788369897344,21788369897344,2.179118,0.016236,12.968740,13.356497,12.414181
2,11097963,300702158,22303765979008,22303765979008,NaN,NaN,20.042427,18.633661,16.980835
3,10308120,252907212,22303765979008,22303765979008,NaN,NaN,20.042427,18.633661,16.980835
4,11082985,299902161,22990960742656,22990960742656,1.914079,0.043403,14.405825,14.929481,13.720057


### Sanity check

In [5]:
print(df.shape[0])

11184628


original small LAMOST df (F or G or K or M dwarf)

In [6]:
def load_votable_to_dataframe(file_path):
    # Read the VOTable file using astropy
    votable = Table.read(file_path, format='votable')

    df = votable.to_pandas()

    return df

file_path = "C:\\git_repo\\cool-dwarf_stellar_parameter_inference_from_survey_data\\data\\LAMOST_data\\LAMOST_G0-9.vot"
df_LAMOST = load_votable_to_dataframe(file_path)
print(df_LAMOST.head())

   COLID_92974          COLID_93001
0    332613222  1192528632157238784
1    332613226  1192594156178353920
2    332613228  1192574674206669824
3    332613231  1192584535451587840
4    332613235  1192581511794617088


In [7]:
# Check the number of rows in the dataframe
num_rows = df_LAMOST.shape[0]
print("Number of rows:", num_rows)

Number of rows: 11441011


In [8]:
# Check if a specific value is in the 'gaia_dr3_id' column
value_to_check = 1192528632157238784
# for F, K, M, it is gaia_dr3_id; for G is it gaia_source_id
is_value_present = value_to_check in df['gaia_dr3_id'].values
print(f"Is the value {value_to_check} in 'gaia_dr3_id' column?", is_value_present)
#is_value_present = value_to_check in df['gaia_source_id'].values
#print(f"Is the value {value_to_check} in 'gaia_source_id' column?", is_value_present)


Is the value 1192528632157238784 in 'gaia_dr3_id' column? True


In [9]:
# Check if a specific value is in the 'gaia_dr3_id' column
value_to_check = 332613231
# for F, K, M, it is gaia_dr3_id; for G is it gaia_source_id
#is_value_present = value_to_check in df['gaia_dr3_id'].values
#print(f"Is the value {value_to_check} in 'gaia_dr3_id' column?", is_value_present)
is_value_present = value_to_check in df['obsid'].values
print(f"Is the value {value_to_check} in 'obsid' column?", is_value_present)


Is the value 332613231 in 'obsid' column? True


### Clean the Gaia X LAMOST_small data

In [10]:
# Drop rows where Parallax is missing or negative (bad data)
df_clean = df.dropna(subset=['obsid', 'Plx', 'Gmag', 'BPmag', 'RPmag'])
df_clean = df_clean[df_clean['Plx'] > 0]  # Parallax must be positive for distance

# Calculate Distance (in parsecs)
# Since Gaia Plx is in milliarcseconds, so we divide 1000 by Plx rather than 1/Plx
df_clean['distance_gaia_pc'] = 1000 / df_clean['Plx']

# Calculate Absolute G Magnitude (M_G)
# Formula: M = m - 5*log10(d) + 5
#df_clean['abs_Gmag'] = df_clean['Gmag'] - 5 * np.log10(df_clean['distance_gaia_pc']) + 5
#df_clean['abs_Bmag'] = df_clean['BPmag'] - 5 * np.log10(df_clean['distance_gaia_pc']) + 5
#df_clean['abs_Rmag'] = df_clean['RPmag'] - 5 * np.log10(df_clean['distance_pc']) + 5

#print(df_clean[['subclass', 'distance_pc', 'abs_Gmag', 'bp_rp']].head())
print(df_clean.head())

   lamost_g_dwarf_oid      obsid  gaia_source_id     gaia_dr3_id       Plx  \
0             1889706  482907076  12751758735104  12751758735104  0.837770   
1             1891666  483007065  21788369897344  21788369897344  2.179118   
4            11082985  299902161  22990960742656  22990960742656  1.914079   
5            10308037  252907071  26048977453312  26048977453312  0.932747   
6             4666290  845004236  26048977453312  26048977453312  0.932747   

      e_Plx       Gmag      BPmag      RPmag  distance_gaia_pc  
0  0.078726  16.563858  17.084002  15.898994       1193.645318  
1  0.016236  12.968740  13.356497  12.414181        458.901162  
4  0.043403  14.405825  14.929481  13.720057        522.444470  
5  0.074583  16.905231  17.494980  16.175350       1072.102140  
6  0.074583  16.905231  17.494980  16.175350       1072.102140  


In [11]:
print(df_clean.shape[0])

10739575


In [12]:
df_clean = df_clean.drop(columns=['lamost_g_dwarf_oid', 'e_Plx', 'Plx', 'gaia_dr3_id'])
#df_clean = df_clean.drop(columns=['Plx', 'gaia_dr3_id'])

In [13]:
df_clean.head()

,obsid,gaia_source_id,Gmag,BPmag,RPmag,distance_gaia_pc
0,482907076,12751758735104,16.563858,17.084002,15.898994,1193.645318
1,483007065,21788369897344,12.968740,13.356497,12.414181,458.901162
4,299902161,22990960742656,14.405825,14.929481,13.720057,522.444470
5,252907071,26048977453312,16.905231,17.494980,16.175350,1072.102140
6,845004236,26048977453312,16.905231,17.494980,16.175350,1072.102140


### Cross match df_clean with full LAMOST and LAMOST_vac dataset

In [14]:
# ...existing code...
import pandas as pd
from astropy.table import Table
from concurrent.futures import ThreadPoolExecutor

# Input Paths
# CHANGE 1: Update the path to your new .vot file
lamost_vot_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\LAMOST_data\LAMOST_G0-9_full.vot"
lamost_fits_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\LAMOST_fits_data\dr8_final_vac_flag_parameter.fits"


In [15]:
# Read the .vot file using astropy
votable = Table.read(lamost_vot_path, format='votable')

# Convert to pandas DataFrame
dataframe = votable.to_pandas()

# Display the first few rows of the DataFrame
dataframe.head(10)

,COLID_92974,COLID_92996,COLID_92997,COLID_92998,COLID_92999,COLID_93000,COLID_93001,COLID_93011,COLID_93019,COLID_93020
0,385812071,17.2724,16.205700,15.740300,15.520300,15.384600,2847297171010496768,NaN,NaN,NaN
1,385812072,-999.0000,-999.000000,17.302099,-999.000000,-999.000000,2847313457526429184,5445.37,4.125,0.143
2,385812073,15.3292,14.763900,14.559800,14.471800,14.398600,2847303252684180992,5352.02,4.449,-0.217
3,385812074,15.3441,14.796100,14.593700,14.506300,14.440000,2847309609235778688,NaN,NaN,NaN
4,385812075,20.2470,-999.000000,-999.000000,-999.000000,-999.000000,2847110563271307264,6265.19,4.168,-0.546
5,385812076,-999.0000,-999.000000,-999.000000,19.642599,-999.000000,2847062116040222848,NaN,NaN,NaN
6,385812077,-999.0000,-999.000000,-999.000000,-999.000000,19.411600,2847086477094792832,NaN,NaN,NaN
7,385812078,16.6544,16.304701,16.175600,16.155600,16.124001,2847066239208824064,NaN,NaN,NaN
8,385812080,-999.0000,-999.000000,18.465599,-999.000000,-999.000000,2847083144200055168,5700.19,4.202,-0.419
9,385812081,-999.0000,-999.000000,18.247200,-999.000000,-999.000000,2847091631055522816,NaN,NaN,NaN


In [16]:

# Helper functions for reading
# CHANGE 2: Define a function to load VOTables using Astropy
def load_vot(path):
    # Reads the VOTable and converts it immediately to a Pandas DataFrame
    return Table.read(path, format='votable').to_pandas()

def load_fits(path):
    return Table.read(path, format='fits').to_pandas()

# Load datasets in parallel (Exploiting threading for I/O bound tasks)
print("Loading datasets in parallel with ThreadPoolExecutor...")
with ThreadPoolExecutor(max_workers=8) as executor:
    # CHANGE 3: Submit the VOT loading function instead of the CSV one
    future_vot = executor.submit(load_vot, lamost_vot_path)
    future_fits = executor.submit(load_fits, lamost_fits_path)
    
    df_LAMOST = future_vot.result()
    df_LAMOST_vac = future_fits.result()

print("Datasets loaded successfully.")
print(f"df_LAMOST shape: {df_LAMOST.shape}")
print(f"df_LAMOST_vac shape: {df_LAMOST_vac.shape}")

Loading datasets in parallel with ThreadPoolExecutor...
Datasets loaded successfully.
df_LAMOST shape: (11441011, 10)
df_LAMOST_vac shape: (7109030, 112)


In [17]:
# Rename columns in df_LAMOST
column_mapping = {
    'COLID_92974': 'obsid',
    'COLID_92992': 'subclass',
    'COLID_92996': 'mag_ps_g',
    'COLID_92997': 'mag_ps_r',
    'COLID_92998': 'mag_ps_i',
    'COLID_92999': 'mag_ps_z',
    'COLID_93000': 'mag_ps_y',
    'COLID_93001': 'gaia_source_id',
    'COLID_93011': 'teff',
    'COLID_93019': 'logg',
    'COLID_93020': 'feh'
}

# Apply the renaming
df_LAMOST.rename(columns=column_mapping, inplace=True)

# Print the updated column names for verification
print("Updated column names:", df_LAMOST.columns.tolist())

Updated column names: ['obsid', 'mag_ps_g', 'mag_ps_r', 'mag_ps_i', 'mag_ps_z', 'mag_ps_y', 'gaia_source_id', 'teff', 'logg', 'feh']


In [ ]:
# Define columns to keep for df_LAMOST_vac
columns_to_keep_vac = ['obsid', 
                       # all absolute magnitudes
                       'A_GG', 'A_BP', 'A_RP', # Gaia EDR3
                       'A_J', 'A_H', 'A_KS', # 2MASS
                       'A_W1', 'A_W2', # WISE
                       'A_BAP', 'A_VAP', 'A_RAP', #APASS
                       'A_GSD', 'A_RSD', 'A_ISD']  # SDSS

# Define columns to drop for df_LAMOST
irrelevant_columns_lamost = ['spid', 'snrr','snri', 'class','subclass', 'ps_id','gaia_source_id', 'gaia_g_mean_mag',
                             'feh' # all NA in this column
                             ] 

# Define columns to check for missing values
# Rows with NaNs in these columns will be dropped
nan_columns_check_lamost = [] 
nan_columns_check_vac = []

# Apply Cleaning
if irrelevant_columns_lamost:
    df_LAMOST.drop(columns=irrelevant_columns_lamost, inplace=True, errors='ignore')

if columns_to_keep_vac:
    df_LAMOST_vac = df_LAMOST_vac[columns_to_keep_vac]

# Drop rows with specific value (-999) in specified columns
columns_to_check_for_value = ['mag_ps_y', 'mag_ps_z', 'mag_ps_i', 'mag_ps_r', 'mag_ps_g'] # apparent magnitudes
for col in columns_to_check_for_value:
    if col in df_LAMOST.columns:
        df_LAMOST = df_LAMOST[df_LAMOST[col] != -999]
# Filter out invalid teff values
# Remove rows where teff < 1000 or teff > 10000
if 'teff' in df_LAMOST.columns:
    df_LAMOST = df_LAMOST[(df_LAMOST['teff'] >= 1000) & (df_LAMOST['teff'] <= 10000)]

if nan_columns_check_lamost:
    df_LAMOST.dropna(subset=nan_columns_check_lamost, inplace=True)

if nan_columns_check_vac:
    df_LAMOST_vac.dropna(subset=nan_columns_check_vac, inplace=True)

print("Cleaning complete")

Cleaning complete


In [19]:
df_LAMOST_vac.head(10)

,obsid,A_GG,A_BP,A_RP,A_J,A_H,A_KS,A_W1,A_W2,A_BAP,A_VAP,A_RAP,A_GSD,A_RSD,A_ISD
0,105704216.0,3.751406,3.881304,2.902264,2.336192,2.156372,1.663003,2.524816,1.010052,3.043267,3.874843,2.984920,2.919531,3.610125,3.552638
1,814404042.0,4.068060,4.360425,4.942953,3.212042,4.168449,2.917255,3.933477,3.162223,6.281215,4.351768,4.687097,4.706560,4.050163,6.133793
2,309907119.0,3.817540,3.999802,2.873478,2.580539,1.899501,2.074955,2.280157,1.881451,4.485433,3.520332,3.793094,4.381365,3.434602,3.553047
3,564506028.0,4.082883,4.700488,3.546080,2.831862,3.216836,2.827840,2.193428,2.779294,5.195091,4.152187,4.562494,4.246381,4.444151,4.589468
4,557708087.0,1.277164,1.984290,0.647106,0.245124,-0.012795,-0.667526,0.023370,-0.435779,2.089422,1.887725,1.910387,2.135766,1.395873,1.528985
5,228605225.0,4.908501,5.517083,4.153411,4.032901,3.515193,3.371768,3.212000,3.304538,6.058166,4.773209,4.693232,5.233153,4.242326,4.134130
6,507005148.0,4.721043,5.804685,4.395596,4.018856,3.903585,3.934841,3.216208,3.275542,6.903694,5.404795,4.855844,4.739204,4.408143,4.957470
7,756711092.0,3.068360,4.554669,2.173767,1.954524,1.686570,1.184146,1.316306,2.600422,3.668927,3.549033,3.549498,2.940325,3.033015,1.547707
8,564508230.0,2.538508,4.394424,3.679119,2.047662,2.201259,2.897748,1.874365,1.385151,5.182198,3.290172,3.144165,3.549851,2.390765,3.044298
9,414402010.0,4.645375,5.539663,4.799868,3.656878,3.637173,3.818758,2.607846,3.200141,5.749143,4.662692,5.437180,4.404426,4.987878,4.183251


In [20]:
df_LAMOST.head(10)

,obsid,mag_ps_g,mag_ps_r,mag_ps_i,mag_ps_z,mag_ps_y,teff,logg
2,385812073,15.329200,14.763900,14.5598,14.471800,14.398600,5352.02,4.449
11,385812084,16.532101,16.208401,16.0912,16.051701,16.012501,6007.79,3.939
12,385812086,15.754600,15.376900,15.2538,15.229400,15.204700,5905.31,4.186
15,385812089,14.887700,14.133800,13.8155,13.664300,13.550100,4770.01,4.644
22,385812101,15.543200,15.057200,14.8845,14.837100,14.788300,5670.60,4.533
23,385812103,15.021000,14.420500,14.1816,14.062700,13.978700,5323.70,4.620
24,385812106,15.508000,14.973300,14.7209,14.607500,14.529700,5047.32,2.120
38,385812123,16.018600,15.763800,15.6845,15.688500,15.658700,6367.34,4.144
39,385812124,15.773200,15.166200,14.9666,14.877100,14.814200,5302.50,4.563
41,385812127,14.821000,14.400500,14.2285,14.166400,14.125500,5608.18,4.311


In [21]:
print(df_LAMOST.shape[0])

2160792


In [22]:
print(df_LAMOST_vac.shape[0])

7109030


### Drop NA

In [23]:
# Drop rows where any column has a missing value in df_LAMOST and df_LAMOST_vac
print("Dropping rows with missing values in df_LAMOST and df_LAMOST_vac...")

# Drop rows with missing values in df_LAMOST
df_LAMOST_cleaned = df_LAMOST.dropna()
print(f"df_LAMOST: Dropped {df_LAMOST.shape[0] - df_LAMOST_cleaned.shape[0]} rows with missing values.")

# Drop rows with missing values in df_LAMOST_vac
df_LAMOST_vac_cleaned = df_LAMOST_vac.dropna()
print(f"df_LAMOST_vac: Dropped {df_LAMOST_vac.shape[0] - df_LAMOST_vac_cleaned.shape[0]} rows with missing values.")

# Display the shapes of the cleaned dataframes
print("Cleaned DataFrame Shapes:")
print(f"df_LAMOST_cleaned: {df_LAMOST_cleaned.shape}")
print(f"df_LAMOST_vac_cleaned: {df_LAMOST_vac_cleaned.shape}")

Dropping rows with missing values in df_LAMOST and df_LAMOST_vac...
df_LAMOST: Dropped 24528 rows with missing values.
df_LAMOST_vac: Dropped 146 rows with missing values.
Cleaned DataFrame Shapes:
df_LAMOST_cleaned: (2136264, 8)
df_LAMOST_vac_cleaned: (7108884, 15)


In [24]:
print("Merging df_clean with df_LAMOST")
# First Inner Join
df_merged_1 = df_clean.merge(df_LAMOST_cleaned, how='inner', on='obsid') 

Merging df_clean with df_LAMOST


In [25]:
print("Merging with df_LAMOST_vac")
# Second inner Join
df_final = df_merged_1.merge(df_LAMOST_vac_cleaned, how='inner', on='obsid')

Merging with df_LAMOST_vac


In [26]:
print(df_final.shape)

(1593577, 27)


In [27]:
print(df_final.columns.tolist())

['obsid', 'gaia_source_id', 'Gmag', 'BPmag', 'RPmag', 'distance_gaia_pc', 'mag_ps_g', 'mag_ps_r', 'mag_ps_i', 'mag_ps_z', 'mag_ps_y', 'teff', 'logg', 'A_GG', 'A_BP', 'A_RP', 'A_J', 'A_H', 'A_KS', 'A_W1', 'A_W2', 'A_BAP', 'A_VAP', 'A_RAP', 'A_GSD', 'A_RSD', 'A_ISD']


In [28]:
output_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\Xmatch_result\Xmatch_gaia_LAMOST_LAMOSTvac_G_dwarf.csv"
df_final.to_csv(output_path, index=False)
print(f"df_final saved to {output_path}")

df_final saved to C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\Xmatch_result\Xmatch_gaia_LAMOST_LAMOSTvac_G_dwarf.csv
